Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
silver_table = f"{catalog}.{silver_schema}.cancellation"
gold_table = f"{catalog}.{gold_schema}.dim_cancellation"

Read silver Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

silver_cancellation_df=(
    spark.read
    .format("delta")
    .table(silver_table)
    .filter(F.col("batch_id")==batch_id)
)

In [0]:
gold_dim_cancellation_df=(
    silver_cancellation_df
    .select(
        "cancellation_key",
        "cancellation_reason"
    )
)

Write DataFrame to gold Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
if not spark.catalog.tableExists(gold_table):

    gold_dim_cancellation_df_write=(
        gold_dim_cancellation_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, gold_table)
    (
        delta_table.alias("t")
        .merge(
            gold_dim_cancellation_df.alias("s"),
            "t.cancellation_key = s.cancellation_key"
        )
        .whenMatchedUpdate(
            set={
                "cancellation_key": "s.cancellation_key",
                "cancellation_reason": "s.cancellation_reason"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )